# Scraper Performance & Benchmarking

This notebook presents the auditable runtime and memory benchmarks on a **Google Colab CPU-only instance**, fulfilling the performance evaluation requirements of Milestone 1:
* **Execution Time:** Total wall time, discovery duration, average time per paper.
* **Memory Footprint:** Peak RAM (RSS), Average RAM consumption.
* **Disk Footprint:** Maximum scratch storage required vs. final output storage.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

perf_candidates = [
    Path("../data/performance.json"),
    Path("./data/performance.json"),
    Path("../../logs/performance.json"),
    Path("../logs/performance.json"),
]
json_path = next((p for p in perf_candidates if p.exists()), None)
if json_path is None:
    raise FileNotFoundError("Cannot locate performance.json in standard paths.")

with open(json_path, "r", encoding="utf-8") as f:
    perf = json.load(f)

print(f"Loaded performance logs from {json_path}")


## 1. Auditable Benchmark Metrics

In [ ]:
benchmark_rows = [
    ("Benchmark Environment", "Google Colab (CPU-Only)"),
    ("Total Pipeline Wall Time", f"{perf['total_wall_time_seconds']:.2f} s"),
    ("Entry Discovery Duration", f"{perf.get('stages', {}).get('discovery', 0):.2f} s"),
    ("Peak RAM Consumption (RSS)", f"{perf['peak_ram_mb']:.2f} MB"),
    ("Average RAM Consumption", f"{perf['avg_ram_mb']:.2f} MB"),
    ("Maximum Disk Storage Required", f"{perf['max_disk_mb']:.2f} MB"),
    ("Final Corpus Storage Size", f"{perf['final_disk_mb']:.2f} MB"),
]

bench_df = pd.DataFrame(benchmark_rows, columns=["Benchmark Metric", "Measurement"])
bench_df


## 2. RAM Consumption Profile Over Time

Continuous memory sampling via `psutil` background thread, tracking RAM throughout discovery, parallel download, and citation querying.

In [ ]:
samples = perf.get("ram_timeseries", [])
if samples:
    times = [s[0] for s in samples]
    mems = [s[1] for s in samples]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(times, mems, color="#2b5c8f", lw=2, label="RAM (RSS)")
    ax.axhline(perf["peak_ram_mb"], color="#d9534f", linestyle="--", label=f"Peak RAM: {perf['peak_ram_mb']:.1f} MB")
    ax.axhline(perf["avg_ram_mb"], color="#f0ad4e", linestyle=":", label=f"Avg RAM: {perf['avg_ram_mb']:.1f} MB")

    ax.set_title("RAM Profile Across End-to-End Scraping", fontsize=13, fontweight="bold")
    ax.set_xlabel("Elapsed Time (seconds)", fontsize=11)
    ax.set_ylabel("Memory (MB)", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


## 3. Storage Footprint Comparison

Compares the maximum temporary disk footprint during downloading and unpacking versus the final compact corpus size.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Peak Disk During Processing", "Final Pruned Corpus Size"],
              [perf["max_disk_mb"], perf["final_disk_mb"]],
              color=["#f0ad4e", "#5cb85c"], width=0.45)

for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.2f} MB",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontweight="bold")

ax.set_title("Disk Footprint: Peak vs Final Storage", fontsize=12, fontweight="bold")
ax.set_ylabel("Storage (MB)", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
